# PCA：用 Ames 房屋特征重新建立坐标轴

本 Notebook 复用前三章的 Ames 数值快照。PCA 只读取八个房屋特征，不读取标签来决定主方向。流程固定为：派生房龄、标准化、SVD、解释方差、投影与重建。

In [1]:
from pathlib import Path
import hashlib
import json
import os

import numpy as np
import pandas as pd

EXPECTED_SHA256 = "763867f46c9a8616d7e7ea7599f4ab1cf408609c8aea06e496e65f9330df20fc"
FEATURES = [
    "overall_quality",
    "first_floor_sqft",
    "second_floor_sqft",
    "living_area_sqft",
    "basement_sqft",
    "garage_cars",
    "garage_area_sqft",
    "house_age_at_sale",
]

def locate_dataset():
    candidates = [
        Path("ames-housing-numeric.csv"),
        Path(os.environ["ML_ATLAS_AMES_DATA_PATH"]) if os.environ.get("ML_ATLAS_AMES_DATA_PATH") else None,
        Path("public/datasets/numerical-methods/ames-housing-numeric.csv"),
    ]
    for candidate in candidates:
        if candidate is not None and candidate.is_file():
            return candidate
    raise FileNotFoundError("Place ames-housing-numeric.csv beside this Notebook or set ML_ATLAS_AMES_DATA_PATH")

dataset_path = locate_dataset()
observed_sha = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
assert observed_sha == EXPECTED_SHA256
ames = pd.read_csv(dataset_path)
ames["house_age_at_sale"] = ames["year_sold"] - ames["year_built"]
print(f"dataset={dataset_path.name} · rows={len(ames)} · sha256={observed_sha[:12]}…")

dataset=ames-housing-numeric.csv · rows=2927 · sha256=763867f46c9a…


## 1. 中心化还不够：先统一量纲

房屋质量是 1–10 分，面积是平方英尺，房龄是年。若只减均值，面积的数值尺度会主导方差。这里对每列做总体标准化 `Z=(F-μ)/σ`，让 PCA 比较相关结构而不是单位大小。

In [2]:
F = ames[FEATURES].to_numpy(dtype=float)
means = F.mean(axis=0)
scales = F.std(axis=0, ddof=0)
assert np.all(scales > 0)
Z = (F - means) / scales

print("shape =", list(Z.shape))
print("max |column mean| =", np.max(np.abs(Z.mean(axis=0))))
print("max |column std - 1| =", np.max(np.abs(Z.std(axis=0, ddof=0) - 1)))

shape = [2927, 8]
max |column mean| = 2.318306500344707e-16
max |column std - 1| = 1.1102230246251565e-16


## 2. SVD 给出主方向和方向强度

`Z = U Σ Vᵀ`。`Vᵀ` 的每一行是一个主方向；`σ²/(n-1)` 是该方向上的样本方差。解释方差比只比较这些方差在总方差中的占比。

In [3]:
U, singular_values, Vt = np.linalg.svd(Z, full_matrices=False)
component_variance = singular_values**2 / (len(Z) - 1)
explained_ratio = component_variance / component_variance.sum()
cumulative_ratio = np.cumsum(explained_ratio)
k90 = int(np.searchsorted(cumulative_ratio, 0.90) + 1)

covariance = Z.T @ Z / (len(Z) - 1)
covariance_eigenvalues = np.linalg.eigvalsh(covariance)[::-1]
spectral_difference = float(np.max(np.abs(component_variance - covariance_eigenvalues)))

print("singular values =", np.round(singular_values, 6).tolist())
print("explained variance ratio =", np.round(explained_ratio, 8).tolist())
print("cumulative ratio =", np.round(cumulative_ratio, 8).tolist())
print("components for at least 90% =", k90)
print(f"max |SVD variance - covariance eigenvalue| = {spectral_difference:.3e}")

singular values = [110.14202, 68.303096, 53.306346, 44.04365, 29.955937, 25.052578, 17.395254, 3.229273]
explained variance ratio = [0.51807587, 0.19923612, 0.12135149, 0.08284263, 0.03832244, 0.02680354, 0.01292257, 0.00044535]
cumulative ratio = [0.51807587, 0.71731199, 0.83866348, 0.92150611, 0.95982855, 0.98663209, 0.99955465, 1.0]
components for at least 90% = 4
max |SVD variance - covariance eigenvalue| = 1.332e-15


## 3. 新坐标轴是原始特征的线性组合

载荷的绝对值越大，表示该原始标准化特征对当前主方向贡献越大。符号只表示轴的方向；整条主方向同时乘以 `-1` 仍是同一个 PCA 轴。

In [4]:
def strongest_loadings(component_index, count=4):
    loading = Vt[component_index]
    order = np.argsort(-np.abs(loading), kind="stable")[:count]
    return [
        {"feature": FEATURES[int(index)], "loading": float(loading[int(index)])}
        for index in order
    ]

pc1_loadings = strongest_loadings(0)
pc2_loadings = strongest_loadings(1)
print("PC1 strongest loadings =", pc1_loadings)
print("PC2 strongest loadings =", pc2_loadings)

PC1 strongest loadings = [{'feature': 'garage_cars', 'loading': 0.40972675193926644}, {'feature': 'garage_area_sqft', 'loading': 0.40789344921400694}, {'feature': 'overall_quality', 'loading': 0.40229719450641116}, {'feature': 'basement_sqft', 'loading': 0.36576737954027}]
PC2 strongest loadings = [{'feature': 'second_floor_sqft', 'loading': 0.7623350466433395}, {'feature': 'living_area_sqft', 'loading': 0.38818250422573164}, {'feature': 'basement_sqft', 'loading': -0.3577920984079251}, {'feature': 'first_floor_sqft', 'loading': -0.34773812241476443}]


## 4. 保留几个主成分，就是保留多少结构

投影 `scores = Z V_k`，重建 `Z_hat = scores V_kᵀ`。下面比较只保留 2 个方向与达到至少 90% 累积解释方差所需方向的标准化重建 RMSE。

In [5]:
def reconstruct(kept_components):
    basis = Vt[:kept_components].T
    scores = Z @ basis
    reconstructed = scores @ basis.T
    rmse = float(np.sqrt(np.mean((Z - reconstructed) ** 2)))
    return scores, reconstructed, rmse

scores_two, reconstructed_two, rmse_two = reconstruct(2)
scores_90, reconstructed_90, rmse_90 = reconstruct(k90)
formula_rmse_two = float(np.sqrt(np.sum(singular_values[2:] ** 2) / Z.size))
rmse_formula_difference = abs(rmse_two - formula_rmse_two)

print(f"2 components: explained = {cumulative_ratio[1]:.8f} · standardized RMSE = {rmse_two:.8f}")
print(f"{k90} components: explained = {cumulative_ratio[k90 - 1]:.8f} · standardized RMSE = {rmse_90:.8f}")
print(f"reconstruction RMSE formula difference = {rmse_formula_difference:.3e}")

2 components: explained = 0.71731199 · standardized RMSE = 0.53168413
4 components: explained = 0.92150611 · standardized RMSE = 0.28016761
reconstruction RMSE formula difference = 1.110e-16


In [6]:
summary = {
    "contractVersion": "numerical-methods-batch-2-v1",
    "outputId": "ames-pca-summary",
    "datasetSha256": EXPECTED_SHA256,
    "rows": int(Z.shape[0]),
    "columns": int(Z.shape[1]),
    "features": FEATURES,
    "singularValues": [float(value) for value in singular_values],
    "explainedVarianceRatio": [float(value) for value in explained_ratio],
    "cumulativeExplainedVariance": [float(value) for value in cumulative_ratio],
    "componentsForAtLeast90Percent": k90,
    "twoComponentExplainedVariance": float(cumulative_ratio[1]),
    "twoComponentStandardizedRmse": rmse_two,
    "k90StandardizedRmse": rmse_90,
    "spectralDifference": spectral_difference,
    "reconstructionFormulaDifference": float(rmse_formula_difference),
    "pc1StrongestLoadings": pc1_loadings,
    "pc2StrongestLoadings": pc2_loadings,
}

output_dir = Path(os.environ.get("ML_ATLAS_NUMERICAL_BATCH2_OUTPUT_DIR", "batch-2-outputs"))
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "ames-pca-summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "contractVersion": "numerical-methods-batch-2-v1",
  "outputId": "ames-pca-summary",
  "datasetSha256": "763867f46c9a8616d7e7ea7599f4ab1cf408609c8aea06e496e65f9330df20fc",
  "rows": 2927,
  "columns": 8,
  "features": [
    "overall_quality",
    "first_floor_sqft",
    "second_floor_sqft",
    "living_area_sqft",
    "basement_sqft",
    "garage_cars",
    "garage_area_sqft",
    "house_age_at_sale"
  ],
  "singularValues": [
    110.14202005872835,
    68.30309561741622,
    53.306345714059326,
    44.04365028288754,
    29.955937493447667,
    25.05257815188605,
    17.395253731198054,
    3.229273492940082
  ],
  "explainedVarianceRatio": [
    0.518075870456838,
    0.19923611508890943,
    0.12135149015146957,
    0.08284263453370773,
    0.0383224372698729,
    0.026803539120958245,
    0.01292256800361974,
    0.00044534537462442093
  ],
  "cumulativeExplainedVariance": [
    0.518075870456838,
    0.7173119855457475,
    0.8386634756972171,
    0.9215061102309249,
    0.95